# Context

Home Credit strives to broaden financial inclusion for the unbanked population by providing a positive and safe borrowing experience. Home Credit makes use of a variety of alternative data--including telco and transactional information--to predict their clients' repayment abilities. Data source from [Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk/overview).

# Objective

Ensure that clients capable of repayment are not rejected and that loans are given with a principal, maturity, and repayment calendar that will empower their clients to be successful.

# Data Description

- sk_id_curr: Loan ID.
- target: Target variable (1 - client with payment difficulties: he/she had late payment more than X days on at least one of the first Y installments of the loan in our sample, 0 - all other cases).
- name_contract_type: Identification if loan is cash or revolving.
- code_gender: gender of client.
- age: age of client.
- flag_own_car: Flag if the client owns a car.
- flag_own_realty: Flag if client owns a house or flat.
- amt_income_total: income of the client.
- amt_credit: credit amount of the loan.
- amt_annuity: loan annuity.
- amt_goods_price: price of the goods for which the loan is given.
- name_housing_type: client current housing situation.
- region_population_relative: normalized population of region where client lives (higher number means more populated region).
- days_employed: days before application since started current employment.
- days_registration: days before application did client change registration.
- days_id_publish: days before the application did client change identity document.
- own_car_age: client's car age.
- region_rating_client: Home Credit's rating of the region where client lives (1,2,3).
- reg_region_not_live_region: Flag if client's permanent address does not match contact address.
- reg_region_not_work_region: Flag if client's permanent address does not match work address.
- live_region_not_work_region: Flag if client's contact address does not match work address.
- ext_source_1: Normalized score from external data source.
- ext_source_2: Normalized score from external data source.
- ext_source_3: Normalized score from external data source.
- apartments_avg: normalized apartments size.
- years_build_mode: normalized modulus of building year built.
- obs_30_cnt_social_circle: Number of observations of client's social surroundings with observable 30 DPD (days past due) default.
- days_last_phone_change: days before appliation client changed phone.
- amt_req_credit_bureau_year: number of enquiries to Credit Bureau about the client one day year.
- family_size_grouped: family size categorized into 4 groups, i.e Alone(1), Small(2-4), Medium(5-6), Large(7-20).
- employment_status: client's employment status.
- education_level: client's highest education level.
- marriage_status: client's marriage status.
- total_loans: client's total loan count.
- active_count: client's total active loan count.
- closed_count: client's total closed loan count.
- bad_debt_count: client's total bad debt loan count.
- sold_credit_count: client's total sold loan count.
- total_past_default: client's total past default count.
- total_active_debt: client's total debt in active.
- total_active_credit: client's total credit in active.
- total_active_overdue: client's total overdue amount.
- max_deliq: client's maximum delinquency severity level (0-5).
- deliq_month: client's number of delinquent months.
- total_deliq_month: client's total months observed.

# Libraries and Functions Required

### Libraries

In [1]:
# import data from SQL server
import psycopg2
import sqlalchemy as sa


import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline


from sklearn.preprocessing import (
    StandardScaler, 
    OneHotEncoder, 
    OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer


from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold, 
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
)


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import xgboost as xgb


from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    ConfusionMatrixDisplay,
)


# set warnings
import warnings
warnings.filterwarnings("ignore")

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)    
# setting the precision of floating numbers to 4 decimal points
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Load data

In [2]:
# Load SQL data into a DataFrame
data = pd.read_sql(
    """
    SELECT *
    FROM analytics.fact_table
    """, engine)

In [3]:
SEED = 27
np.random.seed(SEED)

# Analysis

## Data Overview

In [4]:
df = data.copy()

In [5]:
df.shape

(307511, 45)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.isnull().sum()

sk_id_curr                          0
target                              0
name_contract_type                  0
code_gender                         0
flag_own_car                        0
flag_own_realty                     0
amt_income_total                    0
amt_credit                          0
amt_annuity                        12
amt_goods_price                   278
name_housing_type                   0
region_population_relative          0
days_employed                       0
days_registration                   0
days_id_publish                     0
own_car_age                    202929
region_rating_client                0
reg_region_not_live_region          0
reg_region_not_work_region          0
live_region_not_work_region         0
ext_source_1                   173378
ext_source_2                      660
ext_source_3                    60965
apartments_avg                 156061
years_build_mode               204488
obs_30_cnt_social_circle         1021
days_last_ph

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 45 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   sk_id_curr                   307511 non-null  int64  
 1   target                       307511 non-null  bool   
 2   name_contract_type           307511 non-null  object 
 3   code_gender                  307511 non-null  object 
 4   flag_own_car                 307511 non-null  bool   
 5   flag_own_realty              307511 non-null  bool   
 6   amt_income_total             307511 non-null  float64
 7   amt_credit                   307511 non-null  float64
 8   amt_annuity                  307499 non-null  float64
 9   amt_goods_price              307233 non-null  float64
 10  name_housing_type            307511 non-null  object 
 11  region_population_relative   307511 non-null  float64
 12  days_employed                307511 non-null  int64  
 13 

- A lot of missing value, some are actually 0 valued. Will be treating each column carefully.

In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
sk_id_curr,307511.0000,278180.5186,102790.1753,100002.0000,189145.5000,278202.0000,367142.5000,456255.0000
amt_income_total,307511.0000,168797.9193,237123.1463,25650.0000,112500.0000,147150.0000,202500.0000,117000000.0000
amt_credit,307511.0000,599025.9997,402490.7770,45000.0000,270000.0000,513531.0000,808650.0000,4050000.0000
amt_annuity,307499.0000,27108.5739,14493.7373,1615.5000,16524.0000,24903.0000,34596.0000,258025.5000
amt_goods_price,307233.0000,538396.2074,369446.4605,40500.0000,238500.0000,450000.0000,679500.0000,4050000.0000
region_population_relative,307511.0000,0.0209,0.0138,0.0003,0.0100,0.0188,0.0287,0.0725
days_employed,307511.0000,67724.7421,139443.7518,0.0000,933.0000,2219.0000,5707.0000,365243.0000
days_registration,307511.0000,4986.1203,3522.8863,0.0000,2010.0000,4504.0000,7479.5000,24672.0000
days_id_publish,307511.0000,2994.2024,1509.4504,0.0000,1720.0000,3254.0000,4299.0000,7197.0000
own_car_age,104582.0000,12.0611,11.9448,0.0000,5.0000,9.0000,15.0000,91.0000


In [10]:
num_columns = df.describe(include=np.number).columns
num_columns

Index(['sk_id_curr', 'amt_income_total', 'amt_credit', 'amt_annuity',
       'amt_goods_price', 'region_population_relative', 'days_employed',
       'days_registration', 'days_id_publish', 'own_car_age',
       'region_rating_client', 'ext_source_1', 'ext_source_2', 'ext_source_3',
       'apartments_avg', 'years_build_mode', 'obs_30_cnt_social_circle',
       'days_last_phone_change', 'amt_req_credit_bureau_year', 'age',
       'total_loans', 'active_count', 'closed_count', 'bad_debt_count',
       'sold_credit_count', 'total_past_default', 'total_active_debt',
       'total_active_credit', 'total_active_overdue', 'max_deliq',
       'deliq_month', 'total_deliq_month'],
      dtype='object')

In [11]:
cat_columns = df.describe(include=['object','category','bool']).columns
cat_columns

Index(['target', 'name_contract_type', 'code_gender', 'flag_own_car',
       'flag_own_realty', 'name_housing_type', 'reg_region_not_live_region',
       'reg_region_not_work_region', 'live_region_not_work_region',
       'family_size_grouped', 'employment_status', 'education_level',
       'marriage_status'],
      dtype='object')

## Modelling

### Data Preprocessing

In [12]:
df.isnull().sum()

sk_id_curr                          0
target                              0
name_contract_type                  0
code_gender                         0
flag_own_car                        0
flag_own_realty                     0
amt_income_total                    0
amt_credit                          0
amt_annuity                        12
amt_goods_price                   278
name_housing_type                   0
region_population_relative          0
days_employed                       0
days_registration                   0
days_id_publish                     0
own_car_age                    202929
region_rating_client                0
reg_region_not_live_region          0
reg_region_not_work_region          0
live_region_not_work_region         0
ext_source_1                   173378
ext_source_2                      660
ext_source_3                    60965
apartments_avg                 156061
years_build_mode               204488
obs_30_cnt_social_circle         1021
days_last_ph

In [13]:
df[df['code_gender'] == 'XNA']

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month
35657,141289,False,Revolving loans,XNA,True,True,207000.0000,382500.0000,19125.0000,337500.0000,Municipal apartment,0.0207,10044,10024.0000,3537,13.0000,3,False,False,False,NaN,0.2960,0.4615,NaN,NaN,0.0000,286.0000,1.0000,Small,Employed,Secondary,Married,55.0000,8.0000,3.0000,5.0000,0.0000,0.0000,5.0000,92848.5000,1708954.3350,0.0000,NaN,NaN,NaN
38566,144669,False,Revolving loans,XNA,False,True,157500.0000,270000.0000,13500.0000,225000.0000,House / apartment,0.0264,2797,2241.0000,4659,NaN,2,False,False,False,NaN,0.7092,0.3108,0.0165,NaN,0.0000,493.0000,4.0000,Small,Employed,Secondary,Married,38.0000,7.0000,2.0000,4.0000,0.0000,1.0000,4.0000,251698.5000,661567.5000,0.0000,NaN,NaN,NaN
83382,196708,False,Revolving loans,XNA,False,True,135000.0000,405000.0000,20250.0000,225000.0000,House / apartment,0.0358,1228,183.0000,1671,NaN,2,False,False,False,0.4050,0.6592,0.0770,0.0773,0.8955,7.0000,851.0000,3.0000,Small,Employed,Higher education,Married,29.0000,11.0000,4.0000,7.0000,0.0000,0.0000,7.0000,1621762.4700,3761418.6450,14241.8700,NaN,NaN,NaN
189640,319880,False,Revolving loans,XNA,True,True,247500.0000,540000.0000,27000.0000,900000.0000,House / apartment,0.0358,2293,4099.0000,2326,8.0000,2,False,False,False,0.6530,0.6586,0.3606,0.0278,0.6210,10.0000,1681.0000,6.0000,Small,Employed,Higher education,Civil marriage,26.0000,8.0000,5.0000,3.0000,0.0000,0.0000,3.0000,2745701.0550,4224559.5000,0.0000,NaN,NaN,NaN


- We can drop these rows.

In [14]:
df = df[df.code_gender != 'XNA']

In [15]:
df[df.total_active_debt < 0]

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month
161,100190,False,Cash loans,M,True,False,162000.0000,263686.5000,24781.5000,238500.0000,House / apartment,0.0226,4472,464.0000,4529,3.0000,2,False,False,False,0.5350,0.5859,0.7887,0.3093,0.8563,5.0000,1161.0000,3.0000,Small,Employed,Higher education,Married,38.0000,5.0000,2.0000,3.0000,0.0000,0.0000,3.0000,-77.7600,394335.0000,0.0000,NaN,NaN,NaN
189,100219,False,Cash loans,M,False,True,315000.0000,2250000.0000,83515.5000,2250000.0000,House / apartment,0.0326,1514,64.0000,2793,NaN,1,False,False,False,0.3801,0.6934,NaN,0.4753,NaN,0.0000,926.0000,3.0000,Small,Employed,Higher education,Married,31.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,-88.8300,0.0000,0.0000,NaN,NaN,NaN
324,100372,False,Cash loans,F,False,False,90000.0000,531000.0000,29781.0000,531000.0000,House / apartment,0.0152,365243,2319.0000,4279,NaN,2,False,False,False,NaN,0.6737,0.4651,NaN,NaN,0.0000,0.0000,0.0000,Small,Retired,Secondary,Married,60.0000,8.0000,1.0000,7.0000,0.0000,0.0000,7.0000,-255.5100,872068.4100,0.0000,NaN,NaN,NaN
683,100789,False,Cash loans,F,False,True,112500.0000,813195.0000,24786.0000,702000.0000,House / apartment,0.0106,1048,1187.0000,4819,NaN,3,False,False,False,NaN,0.4931,0.7675,0.1113,0.7648,0.0000,1163.0000,0.0000,Small,Employed,Secondary,Married,45.0000,5.0000,2.0000,3.0000,0.0000,0.0000,3.0000,-603.9450,507783.6000,0.0000,NaN,NaN,NaN
932,101075,False,Cash loans,F,False,True,121500.0000,533668.5000,37273.5000,477000.0000,House / apartment,0.0725,2336,2681.0000,4344,NaN,1,False,False,False,NaN,0.6838,0.6786,0.3464,NaN,0.0000,1151.0000,0.0000,Alone,Employed,Secondary,Single,45.0000,9.0000,1.0000,8.0000,0.0000,0.0000,8.0000,-5117.1300,382997.2950,0.0000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306327,454913,False,Cash loans,M,True,False,112500.0000,1125000.0000,36292.5000,1125000.0000,With parents,0.0073,4907,222.0000,4107,20.0000,3,False,False,False,0.2473,0.4441,0.7544,NaN,NaN,0.0000,1697.0000,2.0000,Small,Employed,Higher education,Married,31.0000,9.0000,3.0000,6.0000,0.0000,0.0000,6.0000,-166.0050,609259.5900,0.0000,NaN,NaN,NaN
306607,455236,False,Cash loans,M,False,False,135000.0000,225000.0000,15660.0000,225000.0000,House / apartment,0.0035,3996,7821.0000,4162,NaN,1,False,False,False,NaN,0.6048,NaN,0.0773,0.8563,0.0000,2641.0000,0.0000,Small,Employed,Secondary,Married,65.0000,3.0000,1.0000,2.0000,0.0000,0.0000,2.0000,-132.7950,287478.0000,0.0000,NaN,NaN,NaN
306807,455464,False,Revolving loans,F,False,False,112500.0000,270000.0000,13500.0000,270000.0000,House / apartment,0.0358,4157,6104.0000,4160,NaN,2,False,False,False,0.3962,0.6798,NaN,NaN,NaN,0.0000,3034.0000,0.0000,Small,Employed,Secondary,Single,34.0000,2.0000,1.0000,1.0000,0.0000,0.0000,1.0000,-1637.2350,238359.6000,0.0000,NaN,NaN,NaN
307334,456059,False,Cash loans,M,False,True,157500.0000,450000.0000,17095.5000,450000.0000,House / apartment,0.0264,3876,0.0000,4406,NaN,2,False,False,False,NaN,0.6620,0.7544,0.2227,NaN,2.0000,1266.0000,3.0000,Small,Employed,Higher education,Married,38.0000,3.0000,0.0000,3.0000,0.0000,0.0000,3.0000,-15.2100,1626207.3000,0.0000,NaN,NaN,NaN


- There are excess debt repayment, resulting to negative values. We will clip this to 0.

In [16]:
df['total_active_debt'] = df['total_active_debt'].clip(lower=0)

We fix the columns that has missing values which supposed to be 0 valued.

In [17]:
fix_columns = ['amt_req_credit_bureau_year','total_loans','active_count','closed_count','bad_debt_count','sold_credit_count','total_past_default',
               'total_active_debt', 'total_active_credit','total_active_overdue','max_deliq','deliq_month','total_deliq_month',
               'obs_30_cnt_social_circle']

In [18]:
df[fix_columns] = df[fix_columns].fillna(0)

In [19]:
df.loc[pd.isna(df['family_size_grouped'])]

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month
41982,148605,False,Revolving loans,M,False,True,450000.0000,675000.0000,33750.0000,NaN,Municipal apartment,0.0152,1161,3265.0000,4489,NaN,2,False,True,True,0.6286,0.7006,NaN,NaN,NaN,3.0000,876.0000,0.0000,None,Employed,Secondary,Unknown,34.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
187348,317181,False,Revolving loans,F,False,True,202500.0000,585000.0000,29250.0000,NaN,House / apartment,0.0313,232,1597.0000,1571,NaN,2,False,False,False,NaN,0.6452,0.6707,0.1031,0.9608,1.0000,654.0000,1.0000,None,Employed,Higher education,Unknown,35.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,24624.0000,324000.0000,0.0000,0.0000,0.0000,0.0000


In [20]:
df.loc[pd.isna(df['days_last_phone_change'])]

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month
15709,118330,False,Cash loans,M,True,True,126000.0000,278613.0000,25911.0000,252000.0000,House / apartment,0.0106,293,4790.0000,1075,21.0000,2,False,False,False,NaN,NaN,NaN,0.1237,NaN,0.0000,NaN,0.0000,Small,Employed,Higher education,Married,23.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


We drop this 3 rows.

In [21]:
df.dropna(axis=0,subset=('family_size_grouped','days_last_phone_change'),inplace=True)

In [22]:
df.groupby('flag_own_car')['own_car_age'].agg(['mean','count', lambda x: x.isna().sum()])

,mean,count,<lambda_0>
flag_own_car,,,
False,NaN,0,202920
True,12.0610,104579,5


- Missing values in `own_car_age` when `flag_own_car` is false simply means no car owned, so will be replaced by 0.

In [23]:
df.loc[df['flag_own_car'] == False, 'own_car_age'] = 0

In [24]:
df.isnull().sum()

sk_id_curr                          0
target                              0
name_contract_type                  0
code_gender                         0
flag_own_car                        0
flag_own_realty                     0
amt_income_total                    0
amt_credit                          0
amt_annuity                        12
amt_goods_price                   276
name_housing_type                   0
region_population_relative          0
days_employed                       0
days_registration                   0
days_id_publish                     0
own_car_age                         5
region_rating_client                0
reg_region_not_live_region          0
reg_region_not_work_region          0
live_region_not_work_region         0
ext_source_1                   173374
ext_source_2                      659
ext_source_3                    60963
apartments_avg                 156059
years_build_mode               204484
obs_30_cnt_social_circle            0
days_last_ph

In [25]:
df.groupby('name_housing_type')['apartments_avg'].agg(['mean','count', lambda x: x.isna().sum()])

,mean,count,<lambda_0>
name_housing_type,,,
Co-op apartment,0.1163,638,484
House / apartment,0.1182,134508,138355
Municipal apartment,0.1089,7141,4040
Office apartment,0.1076,1194,1423
Rented apartment,0.1036,1640,3241
With parents,0.1158,6324,8516


In [26]:
df.groupby('name_housing_type')['years_build_mode'].agg(['mean','count', lambda x: x.isna().sum()])

,mean,count,<lambda_0>
name_housing_type,,,
Co-op apartment,0.7541,392,730
House / apartment,0.7604,91479,181384
Municipal apartment,0.7375,4826,6355
Office apartment,0.7729,843,1774
Rented apartment,0.7622,1075,3806
With parents,0.7651,4405,10435


In [27]:
df[['apartments_avg', 'years_build_mode']].corr()

,apartments_avg,years_build_mode
apartments_avg,1.0000,0.3392
years_build_mode,0.3392,1.0000


In [28]:
both_missing = (df['apartments_avg'].isna() & df['years_build_mode'].isna()).sum()
either_missing = (df['apartments_avg'].isna() | df['years_build_mode'].isna()).sum()

print(f"Both missing: {both_missing}")
print(f"Either missing: {either_missing}")

Both missing: 155014
Either missing: 205529


- We add flag column for `apartments_avg` and `years_build_mode` to keep track the missingness.

In [29]:
df['is_apartments_avg_missing'] = df['apartments_avg'].isna().astype(int)
df['is_years_build_missing'] = df['years_build_mode'].isna().astype(int)

In [30]:
df.isnull().sum()

sk_id_curr                          0
target                              0
name_contract_type                  0
code_gender                         0
flag_own_car                        0
flag_own_realty                     0
amt_income_total                    0
amt_credit                          0
amt_annuity                        12
amt_goods_price                   276
name_housing_type                   0
region_population_relative          0
days_employed                       0
days_registration                   0
days_id_publish                     0
own_car_age                         5
region_rating_client                0
reg_region_not_live_region          0
reg_region_not_work_region          0
live_region_not_work_region         0
ext_source_1                   173374
ext_source_2                      659
ext_source_3                    60963
apartments_avg                 156059
years_build_mode               204484
obs_30_cnt_social_circle            0
days_last_ph

- Construct mean and min column for `ext_source`

In [31]:
df['ext_source_mean'] = df[['ext_source_1', 'ext_source_2', 'ext_source_3']].mean(axis=1)
df['ext_source_min']  = df[['ext_source_1', 'ext_source_2', 'ext_source_3']].min(axis=1)

In [32]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
sk_id_curr,307504.0000,278182.3416,102789.9405,100002.0000,189148.5000,278203.5000,367144.2500,456255.0000
amt_income_total,307504.0000,168796.8009,237125.2214,25650.0000,112500.0000,147150.0000,202500.0000,117000000.0000
amt_credit,307504.0000,599029.4373,402494.1263,45000.0000,270000.0000,513531.0000,808650.0000,4050000.0000
amt_annuity,307492.0000,27108.6421,14493.8635,1615.5000,16524.0000,24903.0000,34596.0000,258025.5000
amt_goods_price,307228.0000,538398.6567,369447.4864,40500.0000,238500.0000,450000.0000,679500.0000,4050000.0000
region_population_relative,307504.0000,0.0209,0.0138,0.0003,0.0100,0.0188,0.0287,0.0725
days_employed,307504.0000,67726.2251,139444.9917,0.0000,933.0000,2219.0000,5707.0000,365243.0000
days_registration,307504.0000,4986.1486,3522.8935,0.0000,2010.0000,4504.0000,7480.0000,24672.0000
days_id_publish,307504.0000,2994.2077,1509.4534,0.0000,1720.0000,3254.0000,4299.0000,7197.0000
own_car_age,307499.0000,4.1019,9.0096,0.0000,0.0000,0.0000,5.0000,91.0000


In [33]:
df[df.days_employed >10000]

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month,is_apartments_avg_missing,is_years_build_missing,ext_source_mean,ext_source_min
8,100011,False,Cash loans,F,False,True,112500.0000,1019610.0000,33826.5000,913500.0000,House / apartment,0.0186,365243,7427.0000,3514,0.0000,2,False,False,False,0.5873,0.2057,0.7517,NaN,NaN,1.0000,0.0000,1.0000,Small,Retired,Secondary,Married,55.0000,4.0000,0.0000,4.0000,0.0000,0.0000,4.0000,0.0000,435228.3000,0.0000,0.0000,0.0000,0.0000,1,1,0.5149,0.2057
11,100015,False,Cash loans,F,False,True,38419.1550,148365.0000,10678.5000,135000.0000,House / apartment,0.0152,365243,5246.0000,2512,0.0000,2,False,False,False,0.7220,0.5552,0.6529,NaN,NaN,0.0000,2396.0000,2.0000,Small,Retired,Secondary,Married,56.0000,4.0000,0.0000,4.0000,0.0000,0.0000,4.0000,0.0000,409495.5000,0.0000,0.0000,0.0000,0.0000,1,1,0.6434,0.5552
23,100027,False,Cash loans,F,False,True,83250.0000,239850.0000,23850.0000,225000.0000,House / apartment,0.0063,365243,9012.0000,3684,0.0000,3,False,False,False,NaN,0.6243,0.6691,0.1443,0.8367,0.0000,795.0000,3.0000,Small,Retired,Secondary,Married,68.0000,3.0000,1.0000,2.0000,0.0000,0.0000,2.0000,376713.0000,625297.5000,0.0000,0.0000,0.0000,0.0000,0,0,0.6467,0.6243
38,100045,False,Cash loans,F,False,True,99000.0000,247275.0000,17338.5000,225000.0000,House / apartment,0.0062,365243,9817.0000,4969,0.0000,2,False,False,False,NaN,0.6508,0.7517,NaN,NaN,0.0000,0.0000,2.0000,Small,Retired,Secondary,Married,65.0000,3.0000,1.0000,2.0000,0.0000,0.0000,2.0000,89351.4600,438054.3900,0.0000,0.0000,0.0000,0.0000,1,1,0.7012,0.6508
43,100050,False,Cash loans,F,False,True,108000.0000,746280.0000,42970.5000,675000.0000,House / apartment,0.0110,365243,5745.0000,4576,0.0000,2,False,False,False,NaN,0.7661,0.6848,0.2186,0.8040,0.0000,491.0000,3.0000,Alone,Retired,Higher education,Single,64.0000,3.0000,0.0000,3.0000,0.0000,0.0000,3.0000,0.0000,742741.3800,0.0000,0.0000,0.0000,201.0000,0,0,0.7255,0.6848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307469,456209,False,Cash loans,F,False,True,202500.0000,703728.0000,29943.0000,607500.0000,House / apartment,0.0313,365243,11976.0000,4171,0.0000,2,False,False,False,NaN,0.1956,0.3606,0.0247,NaN,11.0000,1667.0000,0.0000,Alone,Retired,Secondary,Single,61.0000,10.0000,3.0000,7.0000,0.0000,0.0000,7.0000,460003.5000,2188624.5000,0.0000,1.0000,3.0000,341.0000,0,1,0.2781,0.1956
307483,456227,False,Cash loans,F,False,True,99000.0000,247275.0000,16479.0000,225000.0000,House / apartment,0.0073,365243,11211.0000,4213,0.0000,2,False,False,False,NaN,0.5899,0.5209,0.0082,0.6668,1.0000,1775.0000,5.0000,Alone,Retired,Secondary,Separated,68.0000,4.0000,2.0000,2.0000,0.0000,0.0000,2.0000,156951.0000,797242.5000,0.0000,0.0000,0.0000,0.0000,0,0,0.5554,0.5209
307487,456231,False,Cash loans,M,False,True,117000.0000,1071909.0000,31473.0000,936000.0000,House / apartment,0.0101,365243,5485.0000,4115,0.0000,2,False,False,False,NaN,0.3071,0.2553,NaN,NaN,0.0000,846.0000,8.0000,Small,Retired,Secondary,Married,63.0000,7.0000,3.0000,4.0000,0.0000,0.0000,4.0000,563112.0000,3313435.5000,0.0000,0.0000,0.0000,92.0000,1,1,0.2812,0.2553
307505,456249,False,Cash loans,F,Fa

- `days_employed` has values of 365243, and they appear to be from clients who are pensioners. We replace these values to 0.

In [34]:
df['days_employed'].replace(365243, 0, inplace=True)

In [35]:
df[df.days_employed >20000]

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month,is_apartments_avg_missing,is_years_build_missing,ext_source_mean,ext_source_min


In [36]:
df.isnull().sum()

sk_id_curr                          0
target                              0
name_contract_type                  0
code_gender                         0
flag_own_car                        0
flag_own_realty                     0
amt_income_total                    0
amt_credit                          0
amt_annuity                        12
amt_goods_price                   276
name_housing_type                   0
region_population_relative          0
days_employed                       0
days_registration                   0
days_id_publish                     0
own_car_age                         5
region_rating_client                0
reg_region_not_live_region          0
reg_region_not_work_region          0
live_region_not_work_region         0
ext_source_1                   173374
ext_source_2                      659
ext_source_3                    60963
apartments_avg                 156059
years_build_mode               204484
obs_30_cnt_social_circle            0
days_last_ph

- We fill in the remaining numerical columns with missing values by imputing median after splitting.

### Pipeline

In [37]:
X = df.drop(['sk_id_curr','target'],axis=1)
Y = df['target']

#### Splitting data

In [38]:
# Splitting data into training, validation and test set:
# first we split data into 2 parts, say temporary and test

X_temp, X_test, y_temp, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=SEED, stratify=Y
)

# then we split the temporary set into train and validation

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)

(184502, 47) (61501, 47) (61501, 47)


In [39]:
id_train = df.loc[X_train.index, "sk_id_curr"]
id_val = df.loc[X_val.index, "sk_id_curr"]
id_test  = df.loc[X_test.index, "sk_id_curr"]

id_train = id_train.reset_index(drop=True)
id_val = id_val.reset_index(drop=True)
id_test = id_test.reset_index(drop=True)

In [40]:
print("Shape of Training set : ", X_train.shape)
print("Shape of Validation set : ", X_val.shape)
print("Shape of Test set : ", X_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in validation set:")
print(y_val.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

Shape of Training set :  (184502, 47)
Shape of Validation set :  (61501, 47)
Shape of Test set :  (61501, 47)
Percentage of classes in training set:
target
False   0.9193
True    0.0807
Name: proportion, dtype: float64
Percentage of classes in validation set:
target
False   0.9193
True    0.0807
Name: proportion, dtype: float64
Percentage of classes in test set:
target
False   0.9193
True    0.0807
Name: proportion, dtype: float64


Imbalance preserved.

#### Missing value treatment

In [41]:
imputer = SimpleImputer(strategy="median")

In [42]:
X_train_num = X_train.describe(include=np.number).columns
X_train_cat = X_train.describe(include=['object','category','bool']).columns

In [43]:
# Fit and transform the train data

X_train[X_train_num] = imputer.fit_transform(X_train[X_train_num])

# Transform the validation data
X_val[X_train_num] = imputer.transform(X_val[X_train_num])

# Transform the test data
X_test[X_train_num] = imputer.transform(X_test[X_train_num])

In [44]:
# Checking that no column has missing values in train or test sets
print('X_train')
print(X_train.isna().sum())
print("-" * 30)
print('X_val')
print(X_val.isna().sum())
print("-" * 30)
print('X_test')
print(X_test.isna().sum())

X_train
name_contract_type             0
code_gender                    0
flag_own_car                   0
flag_own_realty                0
amt_income_total               0
amt_credit                     0
amt_annuity                    0
amt_goods_price                0
name_housing_type              0
region_population_relative     0
days_employed                  0
days_registration              0
days_id_publish                0
own_car_age                    0
region_rating_client           0
reg_region_not_live_region     0
reg_region_not_work_region     0
live_region_not_work_region    0
ext_source_1                   0
ext_source_2                   0
ext_source_3                   0
apartments_avg                 0
years_build_mode               0
obs_30_cnt_social_circle       0
days_last_phone_change         0
amt_req_credit_bureau_year     0
family_size_grouped            0
employment_status              0
education_level                0
marriage_status                0
ag

#### Feature engineering

- We now add financial ratios into our analysis

In [45]:
def add_financial_ratios(df):
    ## Ratios
    # Credit-Income ratio, higher -> higher risk, 2-5 medium
    df['credit_income_ratio'] = df['amt_credit'] / df['amt_income_total']
    
    # Annuity-Income ratio
    df['annuity_income_ratio'] = df['amt_annuity'] / df['amt_income_total']
    
    # Loan-Value ratio, if > 1 borrower borrowing extra, higher borrowing higher risk
    df['loan_to_value'] = df['amt_credit'] / df['amt_goods_price']
    
    # Employment-age ratio, higher safer
    df['employment_age_ratio'] = df['days_employed'] / df['age']
    
    # Credit-Utilization ratio, higher -> higher risk, 0.3-0.7 medium
    df['credit_utilization_ratio'] = df['total_active_debt'] / (df['total_active_credit'] + 1)
    
    # Debt-to-Income ratio
    df['debt_income_ratio'] = df['total_active_debt'] / df['amt_income_total']
    
    # Overdue-to-debt ratio
    df['overdue_debt_ratio'] = df['total_active_overdue'] / (df['total_active_debt'] + 1)
    
    # debt-to-credit ratio
    df['debt_credit_ratio'] = df['total_active_debt'] / (df['total_active_credit'] + 1)
    
    # loan exposure ratio
    df['loan_exposure_ratio'] = (df['total_active_debt'] + df['amt_credit']) / df['amt_income_total']
    
    # delinquenct ratio
    df['deliq_ratio'] = df['deliq_month'] / (df['total_deliq_month'] + 1)

    return df

In [46]:
X_train = add_financial_ratios(X_train)
X_val = add_financial_ratios(X_val)
X_test = add_financial_ratios(X_test)

In [47]:
X_train.head()

,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month,is_apartments_avg_missing,is_years_build_missing,ext_source_mean,ext_source_min,credit_income_ratio,annuity_income_ratio,loan_to_value,employment_age_ratio,credit_utilization_ratio,debt_income_ratio,overdue_debt_ratio,debt_credit_ratio,loan_exposure_ratio,deliq_ratio
148091,Cash loans,F,False,True,99000.0000,1125000.0000,32895.0000,1125000.0000,House / apartment,0.0152,1959.0000,8.0000,1750.0000,0.0000,2.0000,False,False,False,0.3485,0.6561,0.6347,0.0876,0.7648,3.0000,685.0000,0.0000,Small,Employed,Higher education,Married,32.0000,4.0000,2.0000,2.0000,0.0000,0.0000,2.0000,77229.4500,1733970.7800,0.0000,0.0000,0.0000,75.0000,1.0000,1.0000,0.5464,0.3485,11.3636,0.3323,1.0000,61.2188,0.0445,0.7801,0.0000,0.0445,12.1437,0.0000
37728,Cash loans,M,True,True,180000.0000,497520.0000,52920.0000,450000.0000,House / apartment,0.0308,3360.0000,4397.0000,2952.0000,1.0000,2.0000,False,False,False,0.5071,0.5662,0.4597,0.0876,0.7648,0.0000,319.0000,0.0000,Small,Employed,Higher education,Married,28.0000,4.0000,2.0000,2.0000,0.0000,0.0000,2.0000,209817.0000,2045745.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.4597,0.4597,2.7640,0.2940,1.1056,120.0000,0.1026,1.1657,0.0000,0.1026,3.9297,0.0000
242410,Revolving loans,F,False,True,148500.0000,315000.0000,15750.0000,315000.0000,House / apartment,0.0188,821.0000,482.0000,772.0000,0.0000,2.0000,False,False,False,0.2528,0.4540,0.4669,0.0876,0.7648,4.0000,541.0000,0.0000,Small,Employed,Secondary,Married,31.0000,11.0000,6.0000,5.0000,0.0000,0.0000,5.0000,2396249.5950,3860819.5500,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.3912,0.2528,2.1212,0.1061,1.0000,26.4839,0.6207,16.1364,0.0000,0.6207,18.2576,0.0000
123738,Cash loans,M,True,True,211500.0000,360000.0000,28570.5000,360000.0000,House / apartment,0.0089,1709.0000,13436.0000,3863.0000,9.0000,2.0000,False,False,False,0.5071,0.3331,0.3202,0.0876,0.7648,0.0000,826.0000,4.0000,Small,Employed,Secondary,Married,59.0000,5.0000,1.0000,4.0000,0.0000,0.0000,4.0000,1577250.0000,2880898.3350,0.0000,1.0000,3.0000,87.0000,1.0000,1.0000,0.3266,0.3202,1.7021,0.1351,1.0000,28.9661,0.5475,7.4574,0.0000,0.5475,9.1596,0.0341
195274,Cash loans,M,True,True,180000.0000,1287000.0000,35392.5000,1287000.0000,House / apartment,0.0313,3183.0000,5613.0000,4130.0000,6.0000,2.0000,False,False,False,0.5071,0.6991,0.5371,0.1031,0.7321,0.0000,0.0000,0.0000,Small,Employed,Higher education,Married,32.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.6991,0.6991,7.1500,0.1966,1.0000,99.4688,0.0000,0.0000,0.0000,0.0000,7.1500,0.0000


In [48]:
X_train.shape

(184502, 57)

In [49]:
print(X_train.isin([np.inf,-np.inf]).any().any())

False


In [50]:
# Checking that no column has missing values in train or test sets
print('X_train')
print(X_train.isna().sum())
print("-" * 30)
print('X_val')
print(X_val.isna().sum())
print("-" * 30)
print('X_test')
print(X_test.isna().sum())

X_train
name_contract_type             0
code_gender                    0
flag_own_car                   0
flag_own_realty                0
amt_income_total               0
amt_credit                     0
amt_annuity                    0
amt_goods_price                0
name_housing_type              0
region_population_relative     0
days_employed                  0
days_registration              0
days_id_publish                0
own_car_age                    0
region_rating_client           0
reg_region_not_live_region     0
reg_region_not_work_region     0
live_region_not_work_region    0
ext_source_1                   0
ext_source_2                   0
ext_source_3                   0
apartments_avg                 0
years_build_mode               0
obs_30_cnt_social_circle       0
days_last_phone_change         0
amt_req_credit_bureau_year     0
family_size_grouped            0
employment_status              0
education_level                0
marriage_status                0
ag

#### Outlier treatment

In [51]:
X_train[X_train['days_employed'] / 365  > X_train['age']]

,name_contract_type,code_gender,flag_own_car,flag_own_realty,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_housing_type,region_population_relative,days_employed,days_registration,days_id_publish,own_car_age,region_rating_client,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,ext_source_1,ext_source_2,ext_source_3,apartments_avg,years_build_mode,obs_30_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_year,family_size_grouped,employment_status,education_level,marriage_status,age,total_loans,active_count,closed_count,bad_debt_count,sold_credit_count,total_past_default,total_active_debt,total_active_credit,total_active_overdue,max_deliq,deliq_month,total_deliq_month,is_apartments_avg_missing,is_years_build_missing,ext_source_mean,ext_source_min,credit_income_ratio,annuity_income_ratio,loan_to_value,employment_age_ratio,credit_utilization_ratio,debt_income_ratio,overdue_debt_ratio,debt_credit_ratio,loan_exposure_ratio,deliq_ratio


In [52]:
X_train['days_employed'].quantile([0.95, 0.99, 0.999])

0.9500    6772.0000
0.9900   10899.9800
0.9990   14311.4990
Name: days_employed, dtype: float64

In [53]:
X_train['days_registration'].quantile([0.95, 0.99, 0.999])

0.9500   11428.9500
0.9900   13907.0000
0.9990   16465.9940
Name: days_registration, dtype: float64

In [54]:
(X_train['days_employed'] > 14379).sum()

np.int64(172)

In [55]:
(X_train['days_registration'] > 16530).sum()

np.int64(177)

In [56]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
amt_income_total,184502.0000,168923.1649,290827.2793,25650.0000,112500.0000,144000.0000,202500.0000,117000000.0000
amt_credit,184502.0000,599816.0967,402183.7009,45000.0000,270000.0000,514777.5000,808650.0000,4050000.0000
amt_annuity,184502.0000,27139.7354,14483.3855,2164.5000,16573.5000,24930.0000,34627.5000,230161.5000
amt_goods_price,184502.0000,538995.6073,368990.2177,45000.0000,238500.0000,450000.0000,679500.0000,4050000.0000
region_population_relative,184502.0000,0.0209,0.0138,0.0003,0.0100,0.0188,0.0287,0.0725
days_employed,184502.0000,1959.9248,2312.9755,0.0000,290.0000,1218.0000,2760.0000,17531.0000
days_registration,184502.0000,4993.7472,3526.9840,0.0000,2014.0000,4506.0000,7498.0000,24672.0000
days_id_publish,184502.0000,2994.6914,1510.4570,0.0000,1721.0000,3255.0000,4300.0000,7197.0000
own_car_age,184502.0000,4.1107,9.0081,0.0000,0.0000,0.0000,5.0000,69.0000
region_rating_client,184502.0000,2.0525,0.5095,1.0000,2.0000,2.0000,2.0000,3.0000


- We cap the following list of numerical columns

In [57]:
cap_cols = [
    # Financial amounts
    'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price',
    # Days columns
    'days_employed', 'days_registration', 'days_last_phone_change',   
    # age
    'own_car_age', 'obs_30_cnt_social_circle',   
    # counts
    'amt_req_credit_bureau_year', 'total_loans', 'active_count', 'closed_count', 'total_past_default', 'deliq_month', 'total_deliq_month', 
    # bureau aggregations
    'total_active_debt', 'total_active_credit', 'total_active_overdue',    
    # ratios
    'credit_income_ratio', 'annuity_income_ratio', 'loan_to_value', 'employment_age_ratio', 'credit_utilization_ratio',
    'debt_income_ratio', 'overdue_debt_ratio', 'debt_credit_ratio', 'loan_exposure_ratio', 'deliq_ratio',
] # 29

In [58]:
# Fit on train only
bounds = {col: X_train[col].quantile(0.99) for col in cap_cols}

# Apply to all
for dataset in [X_train, X_val, X_test]:
    for col, upper in bounds.items():
        dataset[col] = dataset[col].clip(upper=upper)

In [59]:
X_train.shape

(184502, 57)

In [60]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
amt_income_total,184502.0000,165741.8640,82243.6031,25650.0000,112500.0000,144000.0000,202500.0000,450000.0000
amt_credit,184502.0000,597021.0902,391622.5752,45000.0000,270000.0000,514777.5000,808650.0000,1852806.7350
amt_annuity,184502.0000,26966.1039,13662.1717,2164.5000,16573.5000,24930.0000,34627.5000,70132.5000
amt_goods_price,184502.0000,537285.9384,361854.4510,45000.0000,238500.0000,450000.0000,679500.0000,1800000.0000
region_population_relative,184502.0000,0.0209,0.0138,0.0003,0.0100,0.0188,0.0287,0.0725
days_employed,184502.0000,1943.6406,2239.6217,0.0000,290.0000,1218.0000,2760.0000,10899.9800
days_registration,184502.0000,4981.8885,3492.8026,0.0000,2014.0000,4506.0000,7498.0000,13907.0000
days_id_publish,184502.0000,2994.6914,1510.4570,0.0000,1721.0000,3255.0000,4300.0000,7197.0000
own_car_age,184502.0000,4.1078,8.9885,0.0000,0.0000,0.0000,5.0000,64.0000
region_rating_client,184502.0000,2.0525,0.5095,1.0000,2.0000,2.0000,2.0000,3.0000


#### Building Pipeline

In [61]:
# Columns to skip capping — already normalized 0-1
no_cap_cols = ['region_population_relative', 'days_id_publish', 'region_rating_client', 'ext_source_1', 'ext_source_2', 'ext_source_3',
               'apartments_avg', 'years_build_mode', 'age', 'bad_debt_count', 'sold_credit_count', 'max_deliq', 
] # 12

ordinal_cols = ['education_level', 'family_size_grouped'] # 2

ohe_cols = ['code_gender', 'name_contract_type', 'flag_own_car', 'flag_own_realty', 'name_housing_type', 'reg_region_not_live_region',
            'reg_region_not_work_region', 'live_region_not_work_region', 'employment_status', 'marriage_status','is_apartments_avg_missing',
           'is_years_build_missing','ext_source_mean','ext_source_min',
] # 14

cat_cols = ['code_gender', 'name_contract_type', 'flag_own_car', 'flag_own_realty', 'name_housing_type', 'reg_region_not_live_region',
            'reg_region_not_work_region', 'live_region_not_work_region', 'employment_status', 'marriage_status','is_apartments_avg_missing',
           'is_years_build_missing','ext_source_mean','ext_source_min', 'education_level', 'family_size_grouped',
] # 16

# education_order = ['Secondary', 'Higher education', 'Degrees']
# family_size_order = ['Alone', 'Small', 'Medium', 'Large']

In [62]:
ordinal_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

In [63]:
ohe_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=True))
])

In [64]:
col_trans = ColumnTransformer(transformers=[
    ('num_cap', StandardScaler(), cap_cols),
    ('num_nocap', StandardScaler(), no_cap_cols),
    ('ord', ordinal_pipeline, ordinal_cols),
    ('ohe', ohe_pipeline, ohe_cols)
])

### Model building

#### Logit

##### RandomSearchCV

In [65]:
lg = LogisticRegression(class_weight='balanced', random_state=SEED)

In [66]:
param_grid = {
    'C': uniform(0.01,10),
}

In [67]:
RCV_logit = RandomizedSearchCV(estimator=lg, param_distributions=param_grid, n_iter=50, cv=StratifiedKFold(n_splits=5), scoring='roc_auc', n_jobs=-1,verbose=2)

In [68]:
%%time
rs_logit_pipeline = make_pipeline(col_trans, RCV_logit)
rs_logit_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
CPU times: total: 20.8 s
Wall time: 7min 34s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomizedsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

In [69]:
print(RCV_logit.best_params_)
print(RCV_logit.best_score_)

{'C': np.float64(0.14629501631003294)}
0.7278726460695805


##### GridSearchCV

In [70]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
}

In [71]:
GCV_logit = GridSearchCV(estimator=lg, param_grid=param_grid, cv=StratifiedKFold(n_splits=5), scoring='roc_auc', n_jobs=-1,verbose=2)

In [72]:
%%time
gs_logit_pipeline = make_pipeline(col_trans, GCV_logit)
gs_logit_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
CPU times: total: 10.5 s
Wall time: 44.8 s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('gridsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [73]:
print(GCV_logit.best_params_)
print(GCV_logit.best_score_)

{'C': 0.01}
0.7420628989854515


#### Decision Tree

##### RandomSearchCV

In [74]:
dtc = DecisionTreeClassifier(class_weight='balanced', random_state=SEED)

In [75]:
param_grid = {
    'max_depth': randint(5,100),
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': randint(30,150),
    'min_samples_split': randint(20,100),
}

In [76]:
RCV_dtc = RandomizedSearchCV(estimator=dtc, n_iter=50, param_distributions=param_grid, cv=StratifiedKFold(n_splits=5), scoring='roc_auc', n_jobs=-1,verbose=2)

In [77]:
%%time
rs_dtc_pipeline = make_pipeline(col_trans, RCV_dtc)
rs_dtc_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
CPU times: total: 35.6 s
Wall time: 1h 1min 49s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomizedsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

In [78]:
print(RCV_dtc.best_params_)
print(RCV_dtc.best_score_)

{'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 112, 'min_samples_split': 99}
0.7127328285251593


##### GridSearchCV

In [79]:
dtc = DecisionTreeClassifier(class_weight='balanced', min_samples_split=50, random_state=SEED)

In [80]:
param_grid = {
    'max_depth': [10, 15],
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': [30, 90, 150],
}

In [81]:
GCV_dtc = GridSearchCV(estimator=dtc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5), scoring='roc_auc', n_jobs=-1,verbose=2)

In [82]:
%%time
gs_dtc_pipeline = make_pipeline(col_trans, GCV_dtc)
gs_dtc_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
CPU times: total: 22 s
Wall time: 13min 7s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('gridsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [83]:
print(GCV_dtc.best_params_)
print(GCV_dtc.best_score_)

{'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 150}
0.7064567031211186


#### Random Forest

##### RandomSearchCV

In [84]:
rfc = RandomForestClassifier(class_weight= 'balanced', random_state= SEED)

In [85]:
param_grid = {
    'n_estimators': randint(200,400),
    'max_depth': randint(5,20),
    'min_samples_leaf': randint(30,100),
    'max_features': ['sqrt', 'log2'],
    'criterion': ['gini', 'entropy'],
}

In [86]:
RCV_rfc = RandomizedSearchCV(estimator= rfc, n_iter=50,param_distributions= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1,verbose= 2)

In [87]:
%%time
rs_rfc_pipeline = make_pipeline(col_trans, RCV_rfc)
rs_rfc_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
CPU times: total: 18.1 s
Wall time: 18min 1s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomizedsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

In [88]:
print(RCV_rfc.best_params_)
print(RCV_rfc.best_score_)

{'criterion': 'entropy', 'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 48, 'n_estimators': 386}
0.6931423454979744


##### GridSearchCV

In [89]:
rfc = RandomForestClassifier(class_weight= 'balanced', n_estimators=300, random_state= SEED)

In [90]:
param_grid = {
    'max_depth': [10, 15],
    'min_samples_leaf': [30, 50, 90],
    'max_features': ['sqrt', 'log2'],
}

In [91]:
GCV_rfc = GridSearchCV(estimator= rfc, param_grid= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1,verbose= 2)

In [92]:
%%time
gs_rfc_pipeline = make_pipeline(col_trans, GCV_rfc)
gs_rfc_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
CPU times: total: 11.4 s
Wall time: 4min 59s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('gridsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [93]:
print(GCV_rfc.best_params_)
print(GCV_rfc.best_score_)

{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 30}
0.6861814307812191


#### Light GB

##### RandomSearchCV

In [94]:
lgb = LGBMClassifier(scale_pos_weight= 11.35, max_depth= -1, random_state= SEED)

In [95]:
param_grid = {
    'reg_alpha': uniform(0.1, 0.4),
    'reg_lambda': uniform(0.1, 0.4),
    'subsample': uniform(0.6, 0.3),
    'colsample_bytree': uniform(0.6, 0.3),
    'n_estimators': randint(1000,3000),
    'learning_rate': uniform(0.001,0.099),
    'num_leaves': randint(63, 127),
    'min_child_samples': randint(200, 400),
}

In [96]:
RCV_lgb = RandomizedSearchCV(estimator= lgb, n_iter=50, param_distributions= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1, verbose= 2)

In [97]:
%%time
rs_lgb_pipeline = make_pipeline(col_trans, RCV_lgb)
rs_lgb_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Info] Number of positive: 14895, number of negative: 169607
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040264 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6581
[LightGBM] [Info] Number of data points in the train set: 184502, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080731 -> initscore=-2.432458
[LightGBM] [Info] Start training from score -2.432458
CPU times: total: 3min 7s
Wall time: 1h 30min 44s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomizedsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

In [98]:
print(RCV_lgb.best_params_)
print(RCV_lgb.best_score_)

{'colsample_bytree': np.float64(0.7413731492786184), 'learning_rate': np.float64(0.007580277620054793), 'min_child_samples': 330, 'n_estimators': 1355, 'num_leaves': 65, 'reg_alpha': np.float64(0.37736796424815733), 'reg_lambda': np.float64(0.20584645119948686), 'subsample': np.float64(0.8237966688070392)}
0.7585654082398486


##### GridSearchCV

In [99]:
lgb = LGBMClassifier(scale_pos_weight= 11.35, max_depth= -1, num_leaves=63, min_child_samples=100, colsample_bytree=0.8, subsample=0.8, random_state= SEED)

In [100]:
param_grid = {
    'n_estimators': [2000, 3000],
    'learning_rate': [0.003, 0.005, 0.01],
    'reg_alpha': [0.1, 0.3, 0.5],
    'reg_lambda': [0.1, 0.3, 0.5],
}

In [101]:
GCV_lgb = GridSearchCV(estimator= lgb, param_grid= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1, verbose= 2)

In [102]:
%%time
gs_lgb_pipeline = make_pipeline(col_trans, GCV_lgb)
gs_lgb_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[LightGBM] [Info] Number of positive: 14895, number of negative: 169607
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127310 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7167
[LightGBM] [Info] Number of data points in the train set: 184502, number of used features: 357
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080731 -> initscore=-2.432458
[LightGBM] [Info] Start training from score -2.432458
CPU times: total: 7min 18s
Wall time: 1h 46min 34s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('gridsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [103]:
print(GCV_lgb.best_params_)
print(GCV_lgb.best_score_)

{'learning_rate': 0.003, 'n_estimators': 3000, 'reg_alpha': 0.5, 'reg_lambda': 0.5}
0.7577533836543209


#### XGB

##### RandomSearchCV

In [104]:
xgb = XGBClassifier(scale_pos_weight= 11.35,booster= 'gbtree', objective= 'binary:logistic',random_state= SEED, importance_type= 'gain')

In [105]:
param_grid = {
    'n_estimators': randint(1000, 1500),
    'learning_rate': uniform(0.05, 0.02),
    'max_depth': randint(4, 5),
    'min_child_weight': randint(50, 100),
    'gamma': uniform(0, 0.5),
    'max_delta_step': randint(1, 5),
    'colsample_bytree': uniform(0.6, 0.2),
    'subsample': uniform(0.6, 0.2),
    'reg_alpha': uniform(0.1, 0.4),
    'reg_lambda': uniform(0.1, 0.4),
}

In [106]:
RCV_xgb = RandomizedSearchCV(estimator= xgb, n_iter=50, param_distributions= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1,verbose= 2)

In [107]:
%%time
rs_xgb_pipeline = make_pipeline(col_trans, RCV_xgb)
rs_xgb_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
CPU times: total: 14min 32s
Wall time: 2h 16min 19s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('randomizedsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

In [108]:
print(RCV_xgb.best_params_)
print(RCV_xgb.best_score_)

{'colsample_bytree': np.float64(0.656273580774912), 'gamma': np.float64(0.4168770177692283), 'learning_rate': np.float64(0.05438429413040464), 'max_delta_step': 1, 'max_depth': 4, 'min_child_weight': 99, 'n_estimators': 1099, 'reg_alpha': np.float64(0.46241683613164186), 'reg_lambda': np.float64(0.1511202859548365), 'subsample': np.float64(0.75725504676395)}
0.7566972275718685


##### GridSearchCV

In [109]:
xgb = XGBClassifier(scale_pos_weight= 11.35, min_child_weight=80, n_estimators=3000, learning_rate=0.01, max_depth=4, gamma=0.1, max_delta_step=4, reg_lambda=0.1, colsample_bytree=0.8, subsample=0.8,booster= 'gbtree', objective= 'binary:logistic',random_state= SEED, importance_type= 'gain')

In [110]:
param_grid = {
    'reg_alpha': [0.1,0.3,0.5],
}

In [111]:
GCV_xgb = GridSearchCV(estimator= xgb, param_grid= param_grid, cv= StratifiedKFold(n_splits= 5), scoring= 'roc_auc', n_jobs= -1,verbose= 2)

In [112]:
%%time
gs_xgb_pipeline = make_pipeline(col_trans, GCV_xgb)
gs_xgb_pipeline.fit(X_train,y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
CPU times: total: 42min 11s
Wall time: 29min 23s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('gridsearchcv', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_cap', ...), ('num_nocap', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [113]:
print(GCV_xgb.best_params_)
print(GCV_xgb.best_score_)

{'reg_alpha': 0.5}
0.7595125163543016


#### Validation set comparison

In [114]:
models_rs = {
    'Logistic Regression': rs_logit_pipeline,
    'Decision Tree':       rs_dtc_pipeline,
    'Random Forest':       rs_rfc_pipeline,
    'LightGBM':            rs_lgb_pipeline,
    'XGBoost':             rs_xgb_pipeline,
}

models_gs = {
    'Logistic Regression': gs_logit_pipeline,
    'Decision Tree':       gs_dtc_pipeline,
    'Random Forest':       gs_rfc_pipeline,
    'LightGBM':            gs_lgb_pipeline,
    'XGBoost':             gs_xgb_pipeline,
}

print("Model Comparison — Validation Set")
print("=" * 45)
print("RandomSearchCV")
for name, pipeline in models_rs.items():
    val_proba = pipeline.predict_proba(X_val)[:, 1]
    val_auc   = roc_auc_score(y_val, val_proba)
    print(f"{name:25s} Val AUC: {val_auc:.4f}")
print("=" * 45)    
print("GridSearchCV")
for name, pipeline in models_gs.items():
    val_proba = pipeline.predict_proba(X_val)[:, 1]
    val_auc   = roc_auc_score(y_val, val_proba)
    print(f"{name:25s} Val AUC: {val_auc:.4f}")

Model Comparison — Validation Set
RandomSearchCV
Logistic Regression       Val AUC: 0.7359
Decision Tree             Val AUC: 0.7162
Random Forest             Val AUC: 0.7193
LightGBM                  Val AUC: 0.7699
XGBoost                   Val AUC: 0.7689
GridSearchCV
Logistic Regression       Val AUC: 0.7496
Decision Tree             Val AUC: 0.7146
Random Forest             Val AUC: 0.7145
LightGBM                  Val AUC: 0.7684
XGBoost                   Val AUC: 0.7704



#### Test set performance

In [115]:
print("Model Comparison — Test Set")
print("=" * 45)
print("RandomSearchCV")
for name, pipeline in models_rs.items():
    test_proba = pipeline.predict_proba(X_test)[:, 1]
    test_auc   = roc_auc_score(y_test, test_proba)
    print(f"{name:25s} Test AUC: {test_auc:.4f}")
print("=" * 45)
print("GridSearchCV")
for name, pipeline in models_gs.items():
    test_proba = pipeline.predict_proba(X_test)[:, 1]
    test_auc   = roc_auc_score(y_test, test_proba)
    print(f"{name:25s} Test AUC: {test_auc:.4f}")

Model Comparison — Test Set
RandomSearchCV
Logistic Regression       Test AUC: 0.7305
Decision Tree             Test AUC: 0.7147
Random Forest             Test AUC: 0.7143
LightGBM                  Test AUC: 0.7616
XGBoost                   Test AUC: 0.7592
GridSearchCV
Logistic Regression       Test AUC: 0.7445
Decision Tree             Test AUC: 0.7106
Random Forest             Test AUC: 0.7108
LightGBM                  Test AUC: 0.7611
XGBoost                   Test AUC: 0.7615


# Save pipeline and model

In [116]:
import joblib

In [117]:
joblib.dump(imputer, 'imputer.pkl')

print('Imputer saved!')

Imputer saved!


In [118]:
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_val, 'X_val.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_val, 'y_val.pkl')

print('Splited data saved!')

Splited data saved!


In [119]:
joblib.dump(gs_logit_pipeline,'lg_pipeline.pkl')
joblib.dump(gs_xgb_pipeline,'xgb_pipeline.pkl')

print('Pipeline saved!')

Pipeline saved!
